# 🏓 Pickleball Court View v7- 🔴 12 Keypoints trên video- 📦 Ball bbox + Player bbox (chỉ trên sân)- 🏟️ Court polygon filter: loại khán giả/trọng tài- 👥 Team assignment: Near team (A) + Far team (B)- 🔲 Zone-based projection- 🔄 Kalman + Optical Flow- 🎯 Bounce + In/Out- 🗺️ Court View minimap

## 1. Setup

In [ ]:
!nvidia-smi
!pip install ultralytics opencv-python-headless numpy pandas matplotlib filterpy -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
MODEL_DIR = '/content/drive/MyDrive/pickleball_models'
COURT_MODEL = os.path.join(MODEL_DIR, 'court_keypoint_best.pt')
BALL_MODEL = os.path.join(MODEL_DIR, 'ball_tracker_best.pt')
PLAYER_MODEL = os.path.join(MODEL_DIR, 'player_detection_best.pt')
assert os.path.exists(COURT_MODEL), f'Not found: {COURT_MODEL}'
assert os.path.exists(BALL_MODEL), f'Not found: {BALL_MODEL}'
assert os.path.exists(PLAYER_MODEL), f'Not found: {PLAYER_MODEL}'
INPUT_VIDEO = '/content/drive/MyDrive/pickleball_match.mp4'
assert os.path.exists(INPUT_VIDEO), f'Not found: {INPUT_VIDEO}'
print('✅ All files found!')


## 2. Core Code

In [ ]:
import cv2
import numpy as np
import pandas as pd
import time
from collections import deque
from ultralytics import YOLO
from filterpy.kalman import KalmanFilter as KF

# ========== COURT GEOMETRY (mirrored L/R to match model) ==========
COURT_DST = np.array([
    [0,0],[200,0],[400,0],
    [400,300],[200,300],[0,300],
    [0,580],[200,580],[400,580],
    [400,880],[200,880],[0,880],
], dtype=np.float32)
CW, CH = 400, 880
COURT_LINES = [(0,2),(9,11),(3,5),(6,8),(0,11),(2,9),(1,4),(7,10)]
ZONES = {
    'A': [0,1,4,5], 'B': [1,2,3,4],
    'C': [5,4,7,6], 'D': [4,3,8,7],
    'E': [6,7,10,11], 'F': [7,8,9,10],
}
print('✅ Constants loaded')


In [ ]:
# ========== COURT POLYGON FILTER ==========
class CourtPolygonFilter:
    """
    Dùng 4 góc sân (keypoints 0,2,9,11) tạo polygon.
    Chỉ giữ player có chân nằm TRONG polygon (+ margin).
    Loại bỏ khán giả, trọng tài đứng ngoài sân.
    """
    def __init__(self, margin_ratio=0.15):
        self.margin = margin_ratio
        self.polygon = None

    def update(self, kps, conf_thresh=0.3):
        if kps is None or len(kps) < 12:
            return
        corners_idx = [0, 2, 9, 11]  # topleft→topright→botright→botleft (camera view)
        pts = []
        for idx in corners_idx:
            if kps[idx][2] >= conf_thresh and kps[idx][0] > 0:
                pts.append(kps[idx][:2])
        if len(pts) == 4:
            poly = np.array(pts, dtype=np.float32)
            # Expand polygon by margin
            cx, cy = poly.mean(axis=0)
            expanded = []
            for p in poly:
                dx, dy = p[0]-cx, p[1]-cy
                expanded.append([p[0]+dx*self.margin, p[1]+dy*self.margin])
            self.polygon = np.array(expanded, dtype=np.int32)

    def is_on_court(self, foot_pos):
        if self.polygon is None:
            return True  # No polygon = accept all
        return cv2.pointPolygonTest(self.polygon, (float(foot_pos[0]), float(foot_pos[1])), False) >= 0

print('✅ CourtPolygonFilter ready')


In [ ]:
# ========== TEAM ASSIGNMENT ==========
def assign_teams(players, net_y_cam=None):
    """
    Chia 4 players thành 2 teams dựa trên vị trí Y (camera).
    - Near team (gần camera, Y lớn hơn): Team A (xanh dương)
    - Far team (xa camera, Y nhỏ hơn): Team B (cam)

    Nếu < 4 players, dùng giữa frame làm ranh giới.
    """
    if not players:
        return []

    # Sort by foot Y (ascending = far first)
    sorted_p = sorted(enumerate(players), key=lambda x: x[1]['foot'][1])

    if len(sorted_p) >= 4:
        # Top 2 (smallest Y = farthest) = Team B, Bottom 2 = Team A
        for rank, (orig_idx, _) in enumerate(sorted_p):
            if rank < 2:
                players[orig_idx]['team'] = 'B'
                players[orig_idx]['team_label'] = f'B{rank+1}'
            else:
                players[orig_idx]['team'] = 'A'
                players[orig_idx]['team_label'] = f'A{rank-1}'
    elif len(sorted_p) >= 2:
        mid = len(sorted_p) // 2
        for rank, (orig_idx, _) in enumerate(sorted_p):
            if rank < mid:
                players[orig_idx]['team'] = 'B'
                players[orig_idx]['team_label'] = f'B{rank+1}'
            else:
                players[orig_idx]['team'] = 'A'
                players[orig_idx]['team_label'] = f'A{rank-mid+1}'
    else:
        players[0]['team'] = 'A'
        players[0]['team_label'] = 'A1'

    return players

print('✅ Team assignment ready')


In [ ]:
# ========== ZONE PROJECTOR ==========
class ZoneProjector:
    def __init__(self):
        self._zone_Hs = {}; self._zone_polys = {}; self._global_H = None; self._valid = False
    def update(self, kps, conf_thresh=0.3):
        if kps is None or len(kps)<12: return
        v = (kps[:,2]>=conf_thresh)&(kps[:,0]>0)&(kps[:,1]>0)
        if v.sum()>=6:
            H,_ = cv2.findHomography(kps[v,:2].astype(np.float32), COURT_DST[v].astype(np.float32), cv2.RANSAC, 5.0)
            if H is not None and abs(np.linalg.det(H))>1e-6: self._global_H = H
        for name, indices in ZONES.items():
            if all(v[i] for i in indices):
                src = kps[indices,:2].astype(np.float32); dst = COURT_DST[indices].astype(np.float32)
                H_l = cv2.getPerspectiveTransform(src, dst)
                if H_l is not None:
                    self._zone_Hs[name]=H_l; self._zone_polys[name]=src.astype(np.int32)
        self._n_valid = int(v.sum()); self._valid = len(self._zone_Hs)>0 or self._global_H is not None
    def is_reliable(self, kps=None, min_kps=6, min_conf=0.4, min_area=5000):
        if not hasattr(self,'_n_valid') or self._n_valid < min_kps: return False
        if kps is None: return self._valid
        # Check average confidence
        confs = [kps[i][2] for i in range(len(kps)) if kps[i][2]>0.1 and kps[i][0]>0]
        if not confs or np.mean(confs) < min_conf: return False
        # Check if 4 corners form a reasonable quadrilateral
        corners = [0,2,9,11]
        cpts = [(kps[i][0],kps[i][1]) for i in corners if kps[i][2]>0.2 and kps[i][0]>0]
        if len(cpts) >= 4:
            pts = np.array(cpts)
            area = cv2.contourArea(pts.astype(np.float32))
            if area < min_area: return False
        return self._valid
    def find_zone(self, point):
        for name, poly in self._zone_polys.items():
            if cv2.pointPolygonTest(poly, (float(point[0]),float(point[1])), False)>=0: return name
        return None
    def project(self, point):
        if not self._valid: return None
        zone = self.find_zone(point)
        H = self._zone_Hs.get(zone) if zone else None
        if H is None: H = self._global_H
        if H is None: return None
        o = cv2.perspectiveTransform(np.array([[[point[0],point[1]]]],dtype=np.float32), H)
        x,y = float(o[0][0][0]), float(o[0][0][1])
        if x<-80 or x>CW+80 or y<-80 or y>CH+80: return None
        return (x,y)

print('✅ ZoneProjector ready')


In [ ]:
# ========== OPTICAL FLOW ==========
class OpticalFlowEstimator:
    def __init__(self):
        self.lk_params=dict(winSize=(21,21),maxLevel=3,criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,30,0.01))
        self.prev_gray=None
    def estimate_velocity(self, frame, ball_pos):
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
        if self.prev_gray is None or ball_pos is None: self.prev_gray=gray; return None
        cx,cy=ball_pos
        offsets=np.array([[0,0],[-2,-2],[2,-2],[-2,2],[2,2],[-1,0],[1,0],[0,-1],[0,1]],dtype=np.float32)
        pts=(np.array([[cx,cy]],dtype=np.float32)+offsets).reshape(-1,1,2)
        new_pts,status,_=cv2.calcOpticalFlowPyrLK(self.prev_gray,gray,pts,None,**self.lk_params)
        self.prev_gray=gray
        if new_pts is None: return None
        status=status.flatten().astype(bool)
        if not np.any(status): return None
        vel=(new_pts.reshape(-1,2)-pts.reshape(-1,2))[status]
        return tuple(np.median(vel,axis=0).tolist())
    def update_frame(self, frame):
        self.prev_gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)

print('✅ OpticalFlow ready')


In [ ]:
# ========== BOUNCE DETECTOR ==========
class BounceDetector:
    def __init__(self, window=7, min_gap=8):
        self.y_hist=deque(maxlen=window); self.last_bounce=-999; self.min_gap=min_gap; self.bounces=[]
    def _is_hit_not_bounce(self, ball_pos, players):
        if not players: return False
        bx,by=ball_pos
        for p in players:
            x1,y1,x2,y2=p['bbox']; foot_y=y2; bh=y2-y1
            if not (x1-30<=bx<=x2+30): continue
            if by>=foot_y-bh*0.1: return False
            if by>=foot_y-bh*0.35: return True
            if by<foot_y-bh*0.35: return True
        return False
    def update(self, ball_det, frame_idx, players=None):
        if ball_det is None: return False
        self.y_hist.append(ball_det['pos'][1])
        if len(self.y_hist)<5 or frame_idx-self.last_bounce<self.min_gap: return False
        mid=len(self.y_hist)//2; ys=list(self.y_hist); mid_y=ys[mid]
        before,after=ys[:mid],ys[mid+1:]
        if not before or not after: return False
        if mid_y>max(before)-3 and mid_y>max(after)-3:
            if np.mean(before)<mid_y and np.mean(after)<mid_y:
                if players and self._is_hit_not_bounce(ball_det['pos'],players): return False
                self.last_bounce=frame_idx; self.bounces.append({'frame':frame_idx,'pos':ball_det['pos']}); return True
        return False

print('✅ BounceDetector ready')


In [ ]:
# ========== KALMAN FILTERS ==========
class BallKalmanOF:
    def __init__(self, proc_noise=5.0, meas_noise=2.0, max_miss=10, of_weight=0.3):
        self.proc_noise=proc_noise; self.meas_noise=meas_noise; self.max_miss=max_miss; self.of_weight=of_weight
        self.kf=None; self.miss=0; self.ready=False
    def _create_kf(self, x0, y0):
        kf=KF(dim_x=4,dim_z=2); dt=1.0
        kf.F=np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]],dtype=np.float64)
        kf.H=np.array([[1,0,0,0],[0,1,0,0]],dtype=np.float64)
        q=self.proc_noise**2
        kf.Q=np.array([[q*dt**4/4,0,q*dt**3/2,0],[0,q*dt**4/4,0,q*dt**3/2],[q*dt**3/2,0,q*dt**2,0],[0,q*dt**3/2,0,q*dt**2]],dtype=np.float64)
        kf.R=np.eye(2,dtype=np.float64)*self.meas_noise**2; kf.P*=100
        kf.x=np.array([[x0],[y0],[0],[0]],dtype=np.float64); return kf
    def update(self, detection, of_vel=None):
        if detection is not None:
            cx,cy=detection['pos']
            if not self.ready: self.kf=self._create_kf(cx,cy); self.ready=True; self.miss=0; return(cx,cy)
            self.kf.predict(); self.kf.update(np.array([[cx],[cy]])); self.miss=0
        else:
            if not self.ready: return None
            self.miss+=1
            if self.miss>self.max_miss: return None
            if of_vel is not None:
                a=self.of_weight; self.kf.x[2,0]=(1-a)*self.kf.x[2,0]+a*of_vel[0]; self.kf.x[3,0]=(1-a)*self.kf.x[3,0]+a*of_vel[1]
            self.kf.predict()
        return(float(self.kf.x[0,0]),float(self.kf.x[1,0]))

class Court2DKalman:
    def __init__(self, proc_noise=5.0, meas_noise=10.0, max_pred=15):
        self.kf=KF(dim_x=4,dim_z=2); dt=1.0
        self.kf.F=np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]])
        self.kf.H=np.array([[1,0,0,0],[0,1,0,0]])
        self.kf.Q=np.diag([proc_noise,proc_noise,proc_noise*3,proc_noise*3])
        self.kf.R=np.eye(2)*meas_noise; self.kf.P*=200; self.ready=False; self.miss=0; self.max_pred=max_pred
    def predict_or_update(self, meas_2d, is_bounce=False, trust_measurement=False, margin=80, max_jump=120):
        if is_bounce and meas_2d is not None:
            mx,my=meas_2d
            if not self.ready: self.kf.x=np.array([[mx],[my],[0],[0]]); self.ready=True; self.miss=0; return(mx,my)
            self.kf.predict(); self.kf.update([[mx],[my]]); self.miss=0
            return(float(self.kf.x[0,0]),float(self.kf.x[1,0]))
        elif meas_2d is not None and not self.ready:
            mx,my=meas_2d; self.kf.x=np.array([[mx],[my],[0],[0]]); self.ready=True; self.miss=0; return(mx,my)
        elif meas_2d is not None and self.ready and trust_measurement:
            mx,my=meas_2d
            self.kf.predict()
            px,py=float(self.kf.x[0,0]),float(self.kf.x[1,0])
            inside=(-margin<=mx<=CW+margin) and (-margin<=my<=CH+margin)
            close=np.hypot(mx-px,my-py)<=max_jump
            if inside and close:
                self.kf.update([[mx],[my]]); self.miss=0
                return(float(self.kf.x[0,0]),float(self.kf.x[1,0]))
            self.miss+=1
            if self.miss>self.max_pred: return None
            return(px,py)
        else:
            if not self.ready: return None
            self.miss+=1
            if self.miss>self.max_pred: return None
            self.kf.predict(); return(float(self.kf.x[0,0]),float(self.kf.x[1,0]))

def check_in_out(pt, margin=5):
    if pt is None: return None
    x,y=pt; return 'IN' if -margin<=x<=CW+margin and -margin<=y<=CH+margin else 'OUT'

print('✅ Kalman filters ready')


In [ ]:
# ========== DETECTORS ==========
class CourtDetector:
    def __init__(self, path, conf=0.5, every_n=5, smooth=5):
        self.model=YOLO(path); self.conf=conf; self.every_n=every_n
        self._hist=deque(maxlen=smooth); self._last=None; self._n=0
    def detect(self, frame):
        self._n+=1
        if self.every_n>1 and self._n%self.every_n!=0:
            return self._last.copy() if self._last is not None else None
        res=self.model(frame,verbose=False,conf=self.conf,half=True)
        if not res or res[0].keypoints is None or res[0].keypoints.shape[0]==0:
            return self._last.copy() if self._last is not None else None
        idx=0
        if res[0].boxes and len(res[0].boxes.conf)>0: idx=res[0].boxes.conf.argmax().item()
        xy=res[0].keypoints.xy[idx].cpu().numpy(); cf=res[0].keypoints.conf[idx].cpu().numpy()
        kps=np.zeros((len(xy),3),dtype=np.float32); kps[:,:2]=xy; kps[:,2]=cf
        self._hist.append(kps.copy()); self._last=kps.copy()
        if len(self._hist)<=1: return kps
        ws=np.linspace(0.5,1.0,len(self._hist)); ws/=ws.sum()
        sm=np.zeros_like(kps); tw=np.zeros(len(sm))
        for w,k in zip(ws,self._hist):
            v=(k[:,2]>0)&(k[:,0]>0); sm[v,:2]+=w*k[v,:2]; sm[v,2]+=w*k[v,2]; tw[v]+=w
        nz=tw>0; sm[nz,:2]/=tw[nz,None]; sm[nz,2]/=tw[nz]; return sm

class BallDetector:
    def __init__(self, path, conf=0.25, imgsz=640, max_r=0.06):
        self.model=YOLO(path); self.conf=conf; self.imgsz=imgsz; self.max_r=max_r; self._dets=[]; self._last_pos=None
    def detect(self, frame):
        res=self.model(frame,verbose=False,imgsz=self.imgsz,conf=self.conf,half=True)
        if not res or res[0].boxes is None or len(res[0].boxes)==0: self._dets.append(None); return None
        boxes=res[0].boxes; fh,fw=frame.shape[:2]
        valid=[i for i in range(len(boxes)) if (boxes.xyxy[i].cpu().numpy()[2]-boxes.xyxy[i].cpu().numpy()[0])<fw*self.max_r]
        if not valid: self._dets.append(None); return None
        if self._last_pos and len(valid)>1:
            dists=[np.sqrt(((boxes.xyxy[vi].cpu().numpy()[0]+boxes.xyxy[vi].cpu().numpy()[2])/2-self._last_pos[0])**2+((boxes.xyxy[vi].cpu().numpy()[1]+boxes.xyxy[vi].cpu().numpy()[3])/2-self._last_pos[1])**2) for vi in valid]
            best=valid[np.argmin(dists)]
        else: best=valid[np.argmax([boxes.conf[vi].item() for vi in valid])]
        xy=boxes.xyxy[best].cpu().numpy(); cx,cy=float((xy[0]+xy[2])/2),float((xy[1]+xy[3])/2)
        if self._last_pos and np.sqrt((cx-self._last_pos[0])**2+(cy-self._last_pos[1])**2)>max(fw,fh)*0.2: self._dets.append(None); return None
        det={'pos':(cx,cy),'bbox':(int(xy[0]),int(xy[1]),int(xy[2]),int(xy[3])),'conf':boxes.conf[best].item(),
             'ground_anchor':(cx,float(xy[3])),'size':(float(xy[2]-xy[0]),float(xy[3]-xy[1]))}
        self._last_pos=det['pos']; self._dets.append(det); return det
    def rate(self):
        if not self._dets: return 0
        return sum(1 for d in self._dets if d)/len(self._dets)
    def dataframe(self):
        return pd.DataFrame([{'frame':i,'x':d['pos'][0] if d else np.nan,'y':d['pos'][1] if d else np.nan,'conf':d['conf'] if d else 0,'det':d is not None} for i,d in enumerate(self._dets)])

class PlayerDetector:
    def __init__(self, conf=0.5, max_p=6):
        self.model=YOLO(PLAYER_MODEL); self.conf=conf; self.max_p=max_p
    def detect(self, frame, court_filter=None):
        res=self.model(frame,verbose=False,conf=self.conf,half=True)
        if not res or not res[0].boxes or len(res[0].boxes)==0: return []
        ps=[]
        for i in range(min(len(res[0].boxes),15)):
            xy=res[0].boxes.xyxy[i].cpu().numpy()
            c=res[0].boxes.conf[i].item()
            cls_id = int(res[0].boxes.cls[i].item()) if res[0].boxes.cls is not None else 0
            # Chỉ lấy class Person (thường class 0)
            if cls_id != 0: continue
            foot=(float((xy[0]+xy[2])/2), float(xy[3]))
            # Court polygon filter: loại người ngoài sân
            if court_filter and not court_filter.is_on_court(foot):
                continue
            ps.append({'bbox':(int(xy[0]),int(xy[1]),int(xy[2]),int(xy[3])),'conf':c,'foot':foot})
        ps.sort(key=lambda p:p['conf'], reverse=True)
        return ps[:self.max_p]

print('✅ All detectors ready')


In [ ]:
# ========== CASCADE BALL DETECTION ==========
# Cascade: YOLO ball model → YOLO player model (pickleball class) → Classical CV → Trajectory interpolation

class ClassicalBallDetector:
    """
    Classical CV ball detection: color segmentation + contour + Hough circles.
    Used as fallback when both YOLO models miss.
    """
    def __init__(self, min_area=4, max_area=800, min_circ=0.3):
        # Pickleball: bright yellow/green
        self.lower_hsv = np.array([20, 80, 150])
        self.upper_hsv = np.array([55, 255, 255])
        self.min_area = min_area
        self.max_area = max_area
        self.min_circ = min_circ
        self.prev_gray = None

    def detect(self, frame, court_mask=None):
        h, w = frame.shape[:2]
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, self.lower_hsv, self.upper_hsv)

        # Motion mask: difference from previous frame
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if self.prev_gray is not None:
            diff = cv2.absdiff(gray, self.prev_gray)
            _, motion_mask = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
            motion_mask = cv2.dilate(motion_mask, None, iterations=2)
            mask = cv2.bitwise_and(mask, motion_mask)
        self.prev_gray = gray.copy()

        # Court mask: only search within court area
        if court_mask is not None:
            mask = cv2.bitwise_and(mask, court_mask)

        # Morphology cleanup
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        # Contours
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        candidates = []
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < self.min_area or area > self.max_area:
                continue
            perimeter = cv2.arcLength(cnt, True)
            if perimeter == 0: continue
            circularity = 4 * np.pi * area / (perimeter ** 2)
            if circularity < self.min_circ:
                continue
            M = cv2.moments(cnt)
            if M['m00'] == 0: continue
            cx = M['m10'] / M['m00']
            cy = M['m01'] / M['m00']
            # Skip top 20% of frame (usually sky/scoreboard)
            if cy < h * 0.15: continue
            x, y, bw, bh = cv2.boundingRect(cnt)
            candidates.append({
                'pos': (float(cx), float(cy)),
                'bbox': (x, y, x+bw, y+bh),
                'conf': min(0.5, circularity * 0.5 + area / self.max_area * 0.3),
                'area': area,
                'circ': circularity,
                'method': 'classical',
                'ground_anchor': (float(cx), float(cy)),
                'size': (float(bw), float(bh)),
            })
        # Return best candidate (highest circularity)
        if candidates:
            candidates.sort(key=lambda c: c['circ'], reverse=True)
            return candidates[0]
        return None


class TrajectoryInterpolator:
    """
    Polynomial trajectory fitting for ball position prediction.
    When ball is missed for 1-5 frames, predict from recent trajectory.
    """
    def __init__(self, history_len=15, max_predict=5, poly_deg=2):
        self.history = deque(maxlen=history_len)
        self.max_predict = max_predict
        self.poly_deg = poly_deg
        self.miss_count = 0

    def add(self, frame_idx, pos):
        self.history.append((frame_idx, pos[0], pos[1]))
        self.miss_count = 0

    def predict(self, frame_idx):
        self.miss_count += 1
        if self.miss_count > self.max_predict:
            return None
        if len(self.history) < 4:
            return None

        frames = np.array([h[0] for h in self.history])
        xs = np.array([h[1] for h in self.history])
        ys = np.array([h[2] for h in self.history])

        try:
            deg = min(self.poly_deg, len(self.history) - 1)
            # Fit polynomials
            px = np.polyfit(frames, xs, deg)
            py = np.polyfit(frames, ys, deg)
            pred_x = float(np.polyval(px, frame_idx))
            pred_y = float(np.polyval(py, frame_idx))

            # Sanity check: prediction shouldn't be too far from last known
            last = self.history[-1]
            dx = abs(pred_x - last[1])
            dy = abs(pred_y - last[2])
            max_move = 60 * self.miss_count  # max pixels per frame
            if dx > max_move or dy > max_move:
                return None

            return {
                'pos': (pred_x, pred_y),
                'bbox': (int(pred_x-5), int(pred_y-5), int(pred_x+5), int(pred_y+5)),
                'conf': max(0.1, 0.4 - self.miss_count * 0.08),
                'method': 'interpolation',
                'ground_anchor': (pred_x, pred_y),
                'size': (10.0, 10.0),
            }
        except (np.RankWarning, np.linalg.LinAlgError):
            return None

    def reset(self):
        self.history.clear()
        self.miss_count = 0


class CascadeBallDetector:
    """
    Cascade detection: YOLO ball → YOLO player (pickleball) → Classical CV → Trajectory interpolation.
    """
    def __init__(self, ball_model_path, player_model_path,
                 ball_conf=0.15, player_ball_conf=0.20,
                 imgsz=640, max_r=0.06):
        # Stage 1: Dedicated ball YOLO model
        self.ball_model = YOLO(ball_model_path)
        self.ball_conf = ball_conf

        # Stage 2: Player model also detects Pickleball (class 1 usually)
        self.player_model = YOLO(player_model_path)
        self.player_ball_conf = player_ball_conf
        self.ball_class_id = 1  # Will auto-detect

        # Stage 3: Classical CV
        self.classical = ClassicalBallDetector()

        # Stage 4: Trajectory interpolation
        self.interpolator = TrajectoryInterpolator()

        self.imgsz = imgsz
        self.max_r = max_r
        self._dets = []
        self._last_pos = None
        self._methods = {'yolo_ball':0, 'yolo_player':0, 'classical':0, 'interpolation':0, 'miss':0}

    def _filter_boxes(self, boxes, frame_shape):
        fh, fw = frame_shape[:2]
        valid = []
        for i in range(len(boxes)):
            xy = boxes.xyxy[i].cpu().numpy()
            w = xy[2] - xy[0]; h = xy[3] - xy[1]
            if w > fw * self.max_r or h > fh * self.max_r: continue
            if w < 2 or h < 2: continue
            valid.append(i)
        return valid

    def _best_det(self, boxes, valid, frame_shape):
        if not valid: return None
        fh, fw = frame_shape[:2]
        if self._last_pos and len(valid) > 1:
            dists = []
            for vi in valid:
                xy = boxes.xyxy[vi].cpu().numpy()
                cx = (xy[0]+xy[2])/2; cy = (xy[1]+xy[3])/2
                dists.append(np.sqrt((cx-self._last_pos[0])**2+(cy-self._last_pos[1])**2))
            best = valid[np.argmin(dists)]
        else:
            best = valid[np.argmax([boxes.conf[vi].item() for vi in valid])]
        xy = boxes.xyxy[best].cpu().numpy()
        cx, cy = float((xy[0]+xy[2])/2), float((xy[1]+xy[3])/2)
        # Jump filter
        if self._last_pos:
            dist = np.sqrt((cx-self._last_pos[0])**2+(cy-self._last_pos[1])**2)
            if dist > max(fw,fh)*0.25: return None
        return {
            'pos':(cx,cy), 'bbox':(int(xy[0]),int(xy[1]),int(xy[2]),int(xy[3])),
            'conf':boxes.conf[best].item(),
            'ground_anchor':(cx, float(xy[3])),
            'size':(float(xy[2]-xy[0]), float(xy[3]-xy[1])),
        }

    def detect(self, frame, frame_idx=0, court_mask=None):
        det = None

        # === Stage 1: YOLO Ball Model ===
        res = self.ball_model(frame, verbose=False, imgsz=self.imgsz, conf=self.ball_conf, half=True)
        if res and res[0].boxes and len(res[0].boxes) > 0:
            valid = self._filter_boxes(res[0].boxes, frame.shape)
            det = self._best_det(res[0].boxes, valid, frame.shape)
            if det:
                det['method'] = 'yolo_ball'
                self._methods['yolo_ball'] += 1

        # === Stage 2: YOLO Player Model (Pickleball class) ===
        if det is None:
            res2 = self.player_model(frame, verbose=False, imgsz=self.imgsz, conf=self.player_ball_conf, half=True)
            if res2 and res2[0].boxes and len(res2[0].boxes) > 0:
                # Filter for ball class only (not person)
                ball_indices = []
                for i in range(len(res2[0].boxes)):
                    cls_id = int(res2[0].boxes.cls[i].item())
                    if cls_id != 0:  # Not person = likely ball
                        ball_indices.append(i)
                if ball_indices:
                    # Create filtered view
                    valid_balls = [bi for bi in ball_indices if bi in self._filter_boxes(res2[0].boxes, frame.shape) or True]
                    # Manual best selection from ball_indices
                    best_ball = None; best_conf = 0
                    for bi in ball_indices:
                        xy = res2[0].boxes.xyxy[bi].cpu().numpy()
                        w = xy[2]-xy[0]; h = xy[3]-xy[1]
                        if w > frame.shape[1]*self.max_r or h > frame.shape[0]*self.max_r: continue
                        conf = res2[0].boxes.conf[bi].item()
                        cx,cy = float((xy[0]+xy[2])/2), float((xy[1]+xy[3])/2)
                        if self._last_pos:
                            dist = np.sqrt((cx-self._last_pos[0])**2+(cy-self._last_pos[1])**2)
                            if dist > max(frame.shape[1],frame.shape[0])*0.25: continue
                        if conf > best_conf:
                            best_conf = conf; best_ball = bi
                    if best_ball is not None:
                        xy = res2[0].boxes.xyxy[best_ball].cpu().numpy()
                        cx,cy = float((xy[0]+xy[2])/2), float((xy[1]+xy[3])/2)
                        det = {
                            'pos':(cx,cy), 'bbox':(int(xy[0]),int(xy[1]),int(xy[2]),int(xy[3])),
                            'conf':best_conf, 'method':'yolo_player',
                            'ground_anchor':(cx,float(xy[3])),
                            'size':(float(xy[2]-xy[0]),float(xy[3]-xy[1])),
                        }
                        self._methods['yolo_player'] += 1

        # === Stage 3: Classical CV ===
        if det is None:
            det = self.classical.detect(frame, court_mask)
            if det:
                det['method'] = 'classical'
                self._methods['classical'] += 1

        # === Stage 4: Trajectory Interpolation ===
        if det is None:
            det = self.interpolator.predict(frame_idx)
            if det:
                det['method'] = 'interpolation'
                self._methods['interpolation'] += 1

        # === Update state ===
        if det is not None:
            self._last_pos = det['pos']
            self.interpolator.add(frame_idx, det['pos'])
        else:
            self._methods['miss'] += 1
            self.interpolator.miss_count += 1

        self._dets.append(det)
        return det

    def rate(self):
        if not self._dets: return 0
        return sum(1 for d in self._dets if d is not None) / len(self._dets)

    def method_stats(self):
        total = sum(self._methods.values())
        print(f'  Detection methods breakdown:')
        for m, c in sorted(self._methods.items(), key=lambda x: -x[1]):
            bar = '█' * (c * 30 // max(total, 1))
            print(f'    {m:15s}: {c:5d} ({c/max(total,1)*100:5.1f}%) {bar}')

    def dataframe(self):
        rows = []
        for i, d in enumerate(self._dets):
            rows.append({
                'frame': i,
                'x': d['pos'][0] if d else np.nan,
                'y': d['pos'][1] if d else np.nan,
                'conf': d['conf'] if d else 0,
                'det': d is not None,
                'method': d.get('method','') if d else 'miss',
            })
        return pd.DataFrame(rows)

print('✅ CascadeBallDetector ready (4-stage: YOLO ball → YOLO player → Classical CV → Trajectory)')


In [ ]:
# ========== VISUALIZATION ==========
C_BALL=(0,255,255); C_KP=(0,0,255); C_W=(255,255,255)
C_IN=(0,255,0); C_OUT=(0,0,255)
C_TEAM_A=(255,150,50)   # Near team = cam
C_TEAM_B=(50,150,255)   # Far team = xanh duong

def draw_keypoints_only(frame, kps, alpha=0.8):
    ov=frame.copy()
    for i,kp in enumerate(kps):
        if kp[2]<0.3 or kp[0]<=0: continue
        x,y=int(kp[0]),int(kp[1])
        cv2.circle(ov,(x,y),8,C_KP,-1); cv2.circle(ov,(x,y),9,C_W,2)
        cv2.putText(ov,str(i),(x+12,y+5),cv2.FONT_HERSHEY_SIMPLEX,0.5,C_W,2,cv2.LINE_AA)
    return cv2.addWeighted(ov,alpha,frame,1-alpha,0)

def draw_ball_bbox(frame, det):
    if det and det.get('method','')!='interpolation':
        x1,y1,x2,y2=det['bbox']
        cv2.rectangle(frame,(x1,y1),(x2,y2),C_BALL,2)
        lbl=f"Ball {det['conf']:.0%}"
        (tw,th),_=cv2.getTextSize(lbl,cv2.FONT_HERSHEY_SIMPLEX,0.5,1)
        cv2.rectangle(frame,(x1,y1-th-8),(x1+tw+6,y1),C_BALL,-1)
        cv2.putText(frame,lbl,(x1+3,y1-5),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,0),1,cv2.LINE_AA)
    return frame

def draw_bounce_indicator(frame, is_bounce, in_out, ball_det):
    if is_bounce and ball_det:
        x,y=int(ball_det['pos'][0]),int(ball_det['pos'][1])
        color=C_IN if in_out=='IN' else C_OUT
        cv2.circle(frame,(x,y),25,color,3)
        cv2.putText(frame,in_out,(x+30,y+5),cv2.FONT_HERSHEY_SIMPLEX,0.9,color,2,cv2.LINE_AA)
    return frame

def draw_player_bbox(frame, players):
    for p in players:
        x1,y1,x2,y2=p['bbox']
        team=p.get('team','A')
        label=p.get('team_label','?')
        c=C_TEAM_A if team=='A' else C_TEAM_B
        cv2.rectangle(frame,(x1,y1),(x2,y2),c,2)
        if 'foot' in p:
            fx,fy=int(p['foot'][0]),int(p['foot'][1])
            cv2.circle(frame,(fx,fy),4,c,-1)
            cv2.circle(frame,(fx,fy),5,C_W,1)
        cv2.putText(frame,label,(x1,y1-8),cv2.FONT_HERSHEY_SIMPLEX,0.6,c,2,cv2.LINE_AA)
    return frame

def _draw_court_base(size=(150,330)):
    sx,sy=size[0]/CW,size[1]/CH
    pad=8; W=size[0]+pad*2; H=size[1]+pad*2
    cv=np.full((H,W,3),(30,30,30),dtype=np.uint8)
    oy,ox=pad,pad
    cv2.rectangle(cv,(ox,oy),(ox+size[0],oy+size[1]),(25,80,25),-1)
    cv2.rectangle(cv,(ox,oy+int(300*sy)),(ox+size[0],oy+int(580*sy)),(25,60,100),-1)
    for (i,j) in COURT_LINES:
        cv2.line(cv,(ox+int(COURT_DST[i][0]*sx),oy+int(COURT_DST[i][1]*sy)),
                 (ox+int(COURT_DST[j][0]*sx),oy+int(COURT_DST[j][1]*sy)),C_W,1,cv2.LINE_AA)
    net_y=oy+int(440*sy)
    cv2.line(cv,(ox,net_y),(ox+size[0],net_y),(200,200,200),2)
    return cv, ox, oy, sx, sy

def create_triple_minimap(ball_2d, trail_2d, players_2d, all_bounce_pts, latest_bounce, size=(150,330)):
    sx,sy=size[0]/CW,size[1]/CH
    pad=8; mW=size[0]+pad*2; mH=size[1]+pad*2
    title_h=24
    total_H=title_h+mH*3+6
    panel=np.full((total_H,mW,3),(40,40,40),dtype=np.uint8)
    cv2.rectangle(panel,(0,0),(mW,title_h),(50,50,50),-1)
    cv2.putText(panel,'COURT VIEW',(mW//2-42,17),cv2.FONT_HERSHEY_SIMPLEX,0.45,C_W,1,cv2.LINE_AA)

    # === MAP 1: Ball Trail + Players ===
    m1,ox,oy,_,_=_draw_court_base(size)
    cv2.putText(m1,'TRAIL',(ox+2,oy+12),cv2.FONT_HERSHEY_SIMPLEX,0.3,(180,180,180),1)
    if trail_2d and len(trail_2d)>1:
        for k in range(1,len(trail_2d)):
            al=(k+1)/len(trail_2d); t=max(1,int(2*al))
            c=(0,int(140+115*al),int(200+55*al))
            x1=ox+max(0,min(int(trail_2d[k-1][0]*sx),size[0]-1))
            y1=oy+max(0,min(int(trail_2d[k-1][1]*sy),size[1]-1))
            x2=ox+max(0,min(int(trail_2d[k][0]*sx),size[0]-1))
            y2=oy+max(0,min(int(trail_2d[k][1]*sy),size[1]-1))
            cv2.line(m1,(x1,y1),(x2,y2),c,t,cv2.LINE_AA)
    if ball_2d is not None:
        bx=ox+max(0,min(int(ball_2d[0]*sx),size[0]-1))
        by=oy+max(0,min(int(ball_2d[1]*sy),size[1]-1))
        cv2.circle(m1,(bx,by),6,(0,100,100),-1)
        cv2.circle(m1,(bx,by),4,C_BALL,-1); cv2.circle(m1,(bx,by),5,C_W,1)
    for p2d in players_2d:
        px=ox+max(0,min(int(p2d['pos'][0]*sx),size[0]-1))
        py=oy+max(0,min(int(p2d['pos'][1]*sy),size[1]-1))
        c=C_TEAM_A if p2d['team']=='A' else C_TEAM_B
        cv2.circle(m1,(px,py),7,c,-1); cv2.circle(m1,(px,py),8,C_W,1)
        cv2.putText(m1,p2d['label'],(px-6,py+3),cv2.FONT_HERSHEY_SIMPLEX,0.25,C_W,1)

    # === MAP 2: All Bounces ===
    m2,ox2,oy2,_,_=_draw_court_base(size)
    cv2.putText(m2,'ALL BOUNCES',(ox2+2,oy2+12),cv2.FONT_HERSHEY_SIMPLEX,0.3,(180,180,180),1)
    n_in=0; n_out=0
    for bp in all_bounce_pts:
        bx=ox2+max(0,min(int(bp['pos'][0]*sx),size[0]-1))
        by=oy2+max(0,min(int(bp['pos'][1]*sy),size[1]-1))
        color=C_IN if bp['in_out']=='IN' else C_OUT; sz=5
        cv2.line(m2,(bx-sz,by-sz),(bx+sz,by+sz),color,2)
        cv2.line(m2,(bx+sz,by-sz),(bx-sz,by+sz),color,2)
        if bp['in_out']=='IN': n_in+=1
        else: n_out+=1
    cv2.putText(m2,f'IN:{n_in} OUT:{n_out}',(ox2+2,oy2+size[1]-4),cv2.FONT_HERSHEY_SIMPLEX,0.28,C_W,1)

    # === MAP 3: Latest Bounce Only ===
    m3,ox3,oy3,_,_=_draw_court_base(size)
    cv2.putText(m3,'LATEST',(ox3+2,oy3+12),cv2.FONT_HERSHEY_SIMPLEX,0.3,(180,180,180),1)
    if latest_bounce is not None:
        bx=ox3+max(0,min(int(latest_bounce['pos'][0]*sx),size[0]-1))
        by=oy3+max(0,min(int(latest_bounce['pos'][1]*sy),size[1]-1))
        color=C_IN if latest_bounce['in_out']=='IN' else C_OUT
        sz=8
        cv2.line(m3,(bx-sz,by-sz),(bx+sz,by+sz),color,3)
        cv2.line(m3,(bx+sz,by-sz),(bx-sz,by+sz),color,3)
        cv2.circle(m3,(bx,by),12,color,2)
        cv2.putText(m3,latest_bounce['in_out'],(bx+14,by+5),cv2.FONT_HERSHEY_SIMPLEX,0.45,color,2,cv2.LINE_AA)

    # Stack vertically
    y_off=title_h
    panel[y_off:y_off+mH,0:mW]=m1; y_off+=mH+3
    panel[y_off:y_off+mH,0:mW]=m2; y_off+=mH+3
    panel[y_off:y_off+mH,0:mW]=m3
    return panel

def overlay_cv(frame, cv_map, margin=10):
    out=frame.copy(); fh,fw=out.shape[:2]; mh,mw=cv_map.shape[:2]
    x,y=fw-mw-margin,margin
    if y+mh>fh: mh=fh-y-2
    cv2.rectangle(out,(x-2,y-2),(x+mw+2,y+min(mh,cv_map.shape[0])+2),(60,60,60),-1)
    out[y:y+min(mh,cv_map.shape[0]),x:x+mw]=cv_map[:min(mh,cv_map.shape[0]),:mw]
    return out

print('Triple minimap visualization ready')


## 3. Run Pipeline

In [ ]:
OUTPUT_VIDEO='/content/output_v7.mp4'
HEATMAP_PATH='/content/heatmap_v7.png'
CSV_PATH='/content/ball_v7.csv'
COURT_CONF=0.5; BALL_CONF=0.25; PLAYER_CONF=0.5
COURT_EVERY_N=5; PLAYER_EVERY_N=2
MAX_FRAMES=None
print(f'📹 Input: {INPUT_VIDEO}')
print(f'📁 Output: {OUTPUT_VIDEO}')


In [ ]:
print('🔄 Loading models...')
court_det = CourtDetector(COURT_MODEL, conf=COURT_CONF, every_n=COURT_EVERY_N)
ball_det_model = CascadeBallDetector(BALL_MODEL, PLAYER_MODEL, ball_conf=0.15, player_ball_conf=0.20)
player_det = PlayerDetector(conf=PLAYER_CONF)
zone_proj = ZoneProjector()
court_filter = CourtPolygonFilter(margin_ratio=0.15)
ball_kf_cam = BallKalmanOF(proc_noise=5.0, meas_noise=2.0, max_miss=10, of_weight=0.3)
court_kf = Court2DKalman(proc_noise=5.0, meas_noise=10.0, max_pred=15)
bounce_det = BounceDetector(window=7, min_gap=8)
of_est = OpticalFlowEstimator()
heatmap = np.zeros((CH,CW), dtype=np.float32)
all_bounces = []

cap = cv2.VideoCapture(INPUT_VIDEO)
fps=cap.get(cv2.CAP_PROP_FPS); VW=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VH=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
to_proc=MAX_FRAMES or total
print(f'📐 {VW}x{VH} @ {fps:.0f}fps | {total} frames')
writer=cv2.VideoWriter(OUTPUT_VIDEO,cv2.VideoWriter_fourcc(*'mp4v'),fps,(VW,VH))

trail_2d=deque(maxlen=60); last_players=[]; pcnt=0; recent_bounces=deque(maxlen=20)
last_ball_pos_cam=None
t0=time.time(); n=0

while True:
    ret,frame = cap.read()
    if not ret: break
    if MAX_FRAMES and n>=MAX_FRAMES: break

    # 1) Court keypoints
    kps = court_det.detect(frame)

    # 2) Update zone projector + court polygon filter
    zone_proj.update(kps)
    court_filter.update(kps)

    # 3) Ball detection
    ball = ball_det_model.detect(frame, frame_idx=n)

    # 4) Players (filtered by court polygon)
    pcnt+=1
    if pcnt%PLAYER_EVERY_N==0:
        raw_players = player_det.detect(frame, court_filter=court_filter)
        # Keep max 4 players, assign teams
        raw_players = raw_players[:4]
        # Net Y = midpoint between kitchen lines (keypoints row1 & row2)
        net_y_cam = None
        if kps is not None and len(kps) >= 8:
            # Average Y of kitchen keypoints (3,4,5 and 6,7,8)
            ky = [kps[i][1] for i in [3,4,5,6,7,8] if kps[i][2] > 0.3 and kps[i][1] > 0]
            if ky: net_y_cam = np.mean(ky)
        last_players = assign_teams(raw_players, net_y_cam=net_y_cam)

    # 5) Optical Flow
    of_vel = of_est.estimate_velocity(frame, last_ball_pos_cam)
    if ball is not None: of_est.update_frame(frame)

    # 6) Camera Kalman+OF
    cam_pos = ball_kf_cam.update(ball, of_vel)
    if cam_pos: last_ball_pos_cam = cam_pos

    # 7) Bounce detection
    is_bounce = bounce_det.update(ball, n, players=last_players)

    # 8) Ball → 2D
    raw_2d = zone_proj.project(np.array(ball['ground_anchor'])) if ball and zone_proj.is_reliable(kps) else None
    trust_2d = False
    if ball is not None and raw_2d is not None:
        bw,bh = ball.get('size',(0.0,0.0))
        trust_2d = is_bounce or (bh <= max(10.0, VH*0.03))
    smooth_2d = court_kf.predict_or_update(raw_2d, is_bounce=is_bounce, trust_measurement=trust_2d)
    if smooth_2d is not None: trail_2d.append(smooth_2d)

    # 9) In/Out at bounce
    in_out = None
    if is_bounce and smooth_2d is not None:
        in_out=check_in_out(smooth_2d)
        bp={'frame':n,'pos':smooth_2d,'in_out':in_out}
        all_bounces.append(bp); recent_bounces.append(bp)
        bx,by=int(smooth_2d[0]),int(smooth_2d[1])
        if 0<=bx<CW and 0<=by<CH: heatmap[by,bx]+=1

    # 10) DRAW
    out=frame.copy()
    if kps is not None: out=draw_keypoints_only(out,kps)
    out=draw_ball_bbox(out,ball)
    out=draw_bounce_indicator(out,is_bounce,in_out,ball)
    out=draw_player_bbox(out,last_players)

    # 11) Players → 2D with team info
    players_2d=[]
    for p in last_players:
        p2d=zone_proj.project(np.array(p['foot'])) if zone_proj.is_reliable(kps) else None
        if p2d is not None:
            players_2d.append({'pos':p2d,'team':p.get('team','A'),'label':p.get('team_label','?')})
    near_2d = sorted([p for p in players_2d if p['team']=='A'], key=lambda p: p['pos'][0])
    far_2d = sorted([p for p in players_2d if p['team']=='B'], key=lambda p: p['pos'][0])
    for idx, p in enumerate(near_2d, start=1):
        p['label'] = f'A{idx}'
    for idx, p in enumerate(far_2d):
        p['label'] = f'B{len(far_2d)-idx}'

    # 12) Court View
    latest_b = all_bounces[-1] if all_bounces else None
    cv_map=create_triple_minimap(smooth_2d,list(trail_2d),players_2d,all_bounces,latest_b)
    out=overlay_cv(out,cv_map)
    writer.write(out); n+=1

    if n%200==0:
        el=time.time()-t0; spd=n/el; eta=(to_proc-n)/spd if spd>0 else 0
        print(f'  [{n/to_proc*100:5.1f}%] {n}/{to_proc} | {spd:.1f}fps | ETA:{int(eta//60)}m{int(eta%60)}s')

cap.release(); writer.release()
el=time.time()-t0
print(f'\n{"="*55}')
print(f'  ✅ DONE! {n} frames / {el:.1f}s ({n/el:.1f} fps)')
print(f'  🎯 Ball detection: {ball_det_model.rate():.1%}')
print(f'  🏓 Bounces: {len(all_bounces)} (IN:{sum(1 for b in all_bounces if b["in_out"]=="IN")} OUT:{sum(1 for b in all_bounces if b["in_out"]=="OUT")})')
print(f'  📁 {OUTPUT_VIDEO}')
print(f'{"="*55}')




## 3.5. 📊 Chẩn đoán (Diagnostics)Phân tích chi tiết ball detection → projection → minimap accuracy.

In [ ]:
# ========== DIAGNOSTIC: Ball Detection Stats ==========
dets = ball_det_model._dets
total_frames = len(dets)
detected_frames = sum(1 for d in dets if d is not None)
miss_frames = total_frames - detected_frames

print('='*60)
print('  📊 BALL DETECTION DIAGNOSTICS')
print('='*60)
print(f'  Total frames:      {total_frames}')
print(f'  Detected:          {detected_frames} ({detected_frames/total_frames*100:.1f}%)')
print(f'  Missed:            {miss_frames} ({miss_frames/total_frames*100:.1f}%)')
print()

# Consecutive miss streaks
streaks = []
curr_streak = 0
for d in dets:
    if d is None:
        curr_streak += 1
    else:
        if curr_streak > 0:
            streaks.append(curr_streak)
        curr_streak = 0
if curr_streak > 0:
    streaks.append(curr_streak)

if streaks:
    print(f'  Miss streaks:      {len(streaks)} gaps')
    print(f'  Longest gap:       {max(streaks)} frames')
    print(f'  Mean gap:          {np.mean(streaks):.1f} frames')
    print(f'  Gaps > 10 frames:  {sum(1 for s in streaks if s > 10)}')
print()

# Confidence stats
confs = [d['conf'] for d in dets if d is not None]
if confs:
    print(f'  Confidence mean:   {np.mean(confs):.3f}')
    print(f'  Confidence min:    {np.min(confs):.3f}')
    print(f'  Confidence median: {np.median(confs):.3f}')
    print(f'  Conf < 0.3:        {sum(1 for c in confs if c < 0.3)} frames')
    print(f'  Conf < 0.5:        {sum(1 for c in confs if c < 0.5)} frames')
print()

# Ball size stats
sizes = [d.get('size', (0,0)) for d in dets if d is not None and d.get('size')]
if sizes:
    ws = [s[0] for s in sizes]
    hs = [s[1] for s in sizes]
    print(f'  Ball bbox width:   mean={np.mean(ws):.1f}  min={np.min(ws):.1f}  max={np.max(ws):.1f}')
    print(f'  Ball bbox height:  mean={np.mean(hs):.1f}  min={np.min(hs):.1f}  max={np.max(hs):.1f}')
print('='*60)

print()
ball_det_model.method_stats()


In [ ]:
# ========== DIAGNOSTIC: Ball Camera Position Stats ==========
positions = [(i, d['pos']) for i, d in enumerate(dets) if d is not None]

if len(positions) >= 2:
    jumps = []
    for k in range(1, len(positions)):
        fi, pi = positions[k-1]
        fj, pj = positions[k]
        gap = fj - fi  # frame gap
        dist = np.sqrt((pj[0]-pi[0])**2 + (pj[1]-pi[1])**2)
        speed = dist / max(gap, 1)  # pixels per frame
        jumps.append({'from':fi, 'to':fj, 'gap':gap, 'dist':dist, 'speed':speed})

    speeds = [j['speed'] for j in jumps]
    dists = [j['dist'] for j in jumps]
    print('='*60)
    print('  📊 BALL CAMERA-SPACE MOVEMENT')
    print('='*60)
    print(f'  Speed (px/frame):  mean={np.mean(speeds):.1f}  median={np.median(speeds):.1f}  max={np.max(speeds):.1f}')
    print(f'  Distance jumps:    mean={np.mean(dists):.1f}  median={np.median(dists):.1f}  max={np.max(dists):.1f}')
    big_jumps = [j for j in jumps if j['speed'] > 50]
    print(f'  Big jumps (>50px/f): {len(big_jumps)} / {len(jumps)}')
    if big_jumps:
        print(f'    Worst: frame {big_jumps[-1]["from"]}→{big_jumps[-1]["to"]}, dist={big_jumps[-1]["dist"]:.0f}px')
    print('='*60)


In [ ]:
# ========== DIAGNOSTIC: 2D Projection Stats ==========
trail_list = list(trail_2d)
print('='*60)
print('  📊 BALL 2D PROJECTION (MINIMAP)')
print('='*60)
print(f'  Trail points:      {len(trail_list)}')

if trail_list:
    xs = [p[0] for p in trail_list]
    ys = [p[1] for p in trail_list]
    inside = sum(1 for x,y in trail_list if 0<=x<=CW and 0<=y<=CH)
    outside = len(trail_list) - inside
    print(f'  Inside court:      {inside} ({inside/len(trail_list)*100:.1f}%)')
    print(f'  Outside court:     {outside} ({outside/len(trail_list)*100:.1f}%)')
    print(f'  X range:           [{min(xs):.0f}, {max(xs):.0f}]  (court: 0-{CW})')
    print(f'  Y range:           [{min(ys):.0f}, {max(ys):.0f}]  (court: 0-{CH})')
    print()

    # 2D jump analysis
    jumps_2d = []
    for k in range(1, len(trail_list)):
        dx = trail_list[k][0] - trail_list[k-1][0]
        dy = trail_list[k][1] - trail_list[k-1][1]
        dist = np.sqrt(dx**2 + dy**2)
        jumps_2d.append(dist)

    if jumps_2d:
        print(f'  2D jump distance:  mean={np.mean(jumps_2d):.1f}  median={np.median(jumps_2d):.1f}  max={np.max(jumps_2d):.1f}')
        teleports = sum(1 for j in jumps_2d if j > 100)
        print(f'  Teleports (>100):  {teleports} ({teleports/len(jumps_2d)*100:.1f}%)')

print()
print(f'  Bounces detected:  {len(all_bounces)}')
n_in = sum(1 for b in all_bounces if b["in_out"]=="IN")
n_out = sum(1 for b in all_bounces if b["in_out"]=="OUT")
print(f'  IN: {n_in}  |  OUT: {n_out}')

if all_bounces:
    bounce_xs = [b['pos'][0] for b in all_bounces]
    bounce_ys = [b['pos'][1] for b in all_bounces]
    bi = sum(1 for x,y in zip(bounce_xs,bounce_ys) if 0<=x<=CW and 0<=y<=CH)
    print(f'  Bounces inside court: {bi}/{len(all_bounces)}')
    print(f'  Bounce X range:    [{min(bounce_xs):.0f}, {max(bounce_xs):.0f}]')
    print(f'  Bounce Y range:    [{min(bounce_ys):.0f}, {max(bounce_ys):.0f}]')
print('='*60)


In [ ]:
# ========== DIAGNOSTIC: Zone Projection Stats ==========
# Re-analyze detections through zone projector
zone_stats = {'A':0,'B':0,'C':0,'D':0,'E':0,'F':0,'none':0}
proj_success = 0
proj_fail = 0
proj_inside = 0
proj_outside = 0

for d in dets:
    if d is None: continue
    pos = d.get('ground_anchor', d['pos'])
    zone = zone_proj.find_zone(np.array(pos))
    if zone:
        zone_stats[zone] += 1
    else:
        zone_stats['none'] += 1

    p2d = zone_proj.project(np.array(pos))
    if p2d is not None:
        proj_success += 1
        if 0 <= p2d[0] <= CW and 0 <= p2d[1] <= CH:
            proj_inside += 1
        else:
            proj_outside += 1
    else:
        proj_fail += 1

print('='*60)
print('  📊 ZONE PROJECTION ANALYSIS')
print('='*60)
print(f'  Projection success: {proj_success}/{detected_frames} ({proj_success/max(detected_frames,1)*100:.1f}%)')
print(f'  Projection failed:  {proj_fail}/{detected_frames}')
print(f'  Inside court:       {proj_inside}/{proj_success}')
print(f'  Outside court:      {proj_outside}/{proj_success}')
print()
print('  Zone distribution (ball camera-space):')
for z in ['A','B','C','D','E','F','none']:
    bar = '█' * (zone_stats[z] // max(1, max(zone_stats.values()) // 20))
    print(f'    {z:4s}: {zone_stats[z]:5d} {bar}')
print('='*60)


In [ ]:
# ========== DIAGNOSTIC: Per-100-frame Breakdown ==========
import matplotlib.pyplot as plt

chunk = 100
n_chunks = (total_frames + chunk - 1) // chunk
det_rates = []
proj_rates = []

for ci in range(n_chunks):
    start = ci * chunk
    end = min(start + chunk, total_frames)
    chunk_dets = dets[start:end]
    n_det = sum(1 for d in chunk_dets if d is not None)
    det_rates.append(n_det / len(chunk_dets) * 100)

    n_proj = 0
    for d in chunk_dets:
        if d is None: continue
        pos = d.get('ground_anchor', d['pos'])
        p2d = zone_proj.project(np.array(pos))
        if p2d is not None and 0 <= p2d[0] <= CW and 0 <= p2d[1] <= CH:
            n_proj += 1
    proj_rates.append(n_proj / max(1, n_det) * 100 if n_det > 0 else 0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
x = [i*chunk for i in range(n_chunks)]

ax1.bar(x, det_rates, width=chunk*0.8, color='steelblue', alpha=0.8)
ax1.set_ylabel('Detection %')
ax1.set_title('Ball Detection Rate per 100 frames')
ax1.set_ylim(0, 100)
ax1.axhline(y=np.mean(det_rates), color='red', linestyle='--', label=f'Mean: {np.mean(det_rates):.1f}%')
ax1.legend()

ax2.bar(x, proj_rates, width=chunk*0.8, color='green', alpha=0.8)
ax2.set_ylabel('In-court Projection %')
ax2.set_title('Ball Projection Inside Court per 100 frames')
ax2.set_ylim(0, 100)
ax2.axhline(y=np.mean(proj_rates), color='red', linestyle='--', label=f'Mean: {np.mean(proj_rates):.1f}%')
ax2.legend()
ax2.set_xlabel('Frame')

plt.tight_layout()
plt.show()

# Print summary table
print('\nFrame Range  | Detection % | Projection Inside %')
print('-' * 50)
for i in range(min(10, n_chunks)):
    print(f'{i*chunk:5d}-{min((i+1)*chunk, total_frames):5d}  | {det_rates[i]:10.1f}% | {proj_rates[i]:10.1f}%')
if n_chunks > 10:
    print(f'  ... ({n_chunks-10} more chunks)')


In [ ]:
# ========== DIAGNOSTIC: Raw vs Projected scatter ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Camera positions
cam_xs = [d['pos'][0] for d in dets if d is not None]
cam_ys = [d['pos'][1] for d in dets if d is not None]
axes[0].scatter(cam_xs, cam_ys, s=2, alpha=0.3, c='blue')
axes[0].set_title(f'Camera Space ({len(cam_xs)} pts)')
axes[0].set_xlabel('X (pixels)'); axes[0].set_ylabel('Y (pixels)')
axes[0].invert_yaxis()
axes[0].set_aspect('equal')

# 2. Raw 2D projections
raw_2d_pts = []
for d in dets:
    if d is None: continue
    pos = d.get('ground_anchor', d['pos'])
    p2d = zone_proj.project(np.array(pos))
    if p2d is not None:
        raw_2d_pts.append(p2d)

if raw_2d_pts:
    rx = [p[0] for p in raw_2d_pts]
    ry = [p[1] for p in raw_2d_pts]
    axes[1].scatter(rx, ry, s=2, alpha=0.3, c='green')
    # Draw court boundary
    axes[1].plot([0,CW,CW,0,0], [0,0,CH,CH,0], 'r-', linewidth=2)
    axes[1].set_title(f'Raw 2D Projection ({len(raw_2d_pts)} pts)')
    axes[1].set_xlabel('X (court)'); axes[1].set_ylabel('Y (court)')
    axes[1].set_aspect('equal')

# 3. Filtered trail (after Kalman)
if trail_list:
    tx = [p[0] for p in trail_list]
    ty = [p[1] for p in trail_list]
    axes[2].scatter(tx, ty, s=2, alpha=0.3, c='orange')
    axes[2].plot([0,CW,CW,0,0], [0,0,CH,CH,0], 'r-', linewidth=2)
    axes[2].set_title(f'Kalman Filtered Trail ({len(trail_list)} pts)')
    axes[2].set_xlabel('X (court)'); axes[2].set_ylabel('Y (court)')
    axes[2].set_aspect('equal')

plt.suptitle('Ball Position Analysis: Camera → Raw 2D → Kalman', fontsize=14)
plt.tight_layout()
plt.show()


## 4. Results

In [ ]:
import matplotlib.pyplot as plt
hm=cv2.GaussianBlur(heatmap,(31,31),0)
if hm.max()>0:
    hm_c=cv2.applyColorMap((hm/hm.max()*255).astype(np.uint8),cv2.COLORMAP_JET)
    cv2.imwrite(HEATMAP_PATH,hm_c)
else: hm_c=np.zeros((CH,CW,3),dtype=np.uint8)
df=ball_det_model.dataframe(); df.to_csv(CSV_PATH,index=False)
fig,axes=plt.subplots(1,2,figsize=(14,8))
axes[0].imshow(cv2.cvtColor(hm_c,cv2.COLOR_BGR2RGB))
axes[0].set_title('Shot Placement Heatmap',fontsize=13); axes[0].axis('off')
w=100; df['r']=df['det'].rolling(w,min_periods=1).mean()
axes[1].plot(df['frame'],df['r'],color='green')
axes[1].set_title(f'Detection Rate (rolling {w})',fontsize=13); axes[1].set_ylim(0,1); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
cap=cv2.VideoCapture(OUTPUT_VIDEO); tot=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
samps=[50,tot//5,2*tot//5,3*tot//5,4*tot//5,max(0,tot-30)]
fig,axes=plt.subplots(2,3,figsize=(21,12))
for idx,fn in enumerate(samps):
    ax=axes[idx//3][idx%3]; cap.set(cv2.CAP_PROP_POS_FRAMES,fn)
    ret,f=cap.read()
    if ret: ax.imshow(cv2.cvtColor(f,cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {fn}'); ax.axis('off')
cap.release()
plt.suptitle('v7 — Court Filter + Team Assignment',fontsize=15)
plt.tight_layout(); plt.show()


## 5. Download

In [ ]:
import shutil
SAVE='/content/drive/MyDrive/pickleball_results'
os.makedirs(SAVE,exist_ok=True)
shutil.copy2(OUTPUT_VIDEO,os.path.join(SAVE,'output_v7.mp4'))
shutil.copy2(HEATMAP_PATH,os.path.join(SAVE,'heatmap_v7.png'))
shutil.copy2(CSV_PATH,os.path.join(SAVE,'ball_v7.csv'))
pd.DataFrame(all_bounces).to_csv(os.path.join(SAVE,'bounces_v7.csv'),index=False)
print(f'✅ Saved: {SAVE}')


In [ ]:
from google.colab import files
files.download(OUTPUT_VIDEO)
